06/03/2026

Mik va a intentar hacer una red convolucional cn pytorch, lol

Estoy utilizando el env dl2024 

- Cloth**Dataset** guarda la info d vertices x caracteristicas () al acceder a estos items con el **DataLoader** le añade la otra dimension d frames(batchsize) para crear el tensor3D
- Redondear valores para optimizar (ahorra memoria)

Ahora el modelo itera con distintos batchSizes en modo shuffle, para ir entrenandose poco a poco. No tiene memoria, pero como guardamos las velocidades y tal probablemente funcione?
> Your model assumes that the current state of the cloth is all it needs to predict the next state (this is called a Markov assumption). In this setup, the network looks at a single frame's positions and velocities and predicts the displacements. Graph Neural Networks (GNNs) or standard Multi-Layer Perceptrons (MLPs) usually take data in this exact shape.

Otra idea sería:
> When to add a frame dimension (Sequence modeling): If your model needs temporal history—meaning it needs to look at, say, the last 5 frames to figure out what happens in the 6th frame. If you were using an LSTM, RNN, or a Spatiotemporal Transformer, your tensor would need to look like [Batch, Sequence_Length, Vertices, Features].
Por ahora no.

**Links Utilizados:**
- https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html
- https://lixiaoguang.medium.com/build-cnn-from-scratch-5-convolutional-neural-network-86b4d0323fb0

In [1]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import io
import torch
import os, os.path

# WORKING WITH 
datasetPath = 'data/'

def loadAndMergeCSV(csvRoute):
    """
    Carga de los CSV y mergeo en un único CSV. Todos los CSV estarán en la ruta 'data/', y se excluirá el CSV
    'mergedCSV.csv', producto de los mergeos si se ejecutase antes
    """
    csvRoute= 'data/'
    finalData = pd.DataFrame()
    for csvfile in [f for f in os.listdir(csvRoute) if os.path.isfile(csvRoute + f)]:
        if (csvfile != 'mergedCSV.csv' and os.path.splitext(csvfile)[1] == '.csv'):
            data = pd.read_csv(csvRoute + csvfile)
            finalData = pd.concat([data, finalData], ignore_index=True)

    return finalData

cloth_info = loadAndMergeCSV(datasetPath)

print('cloth_info shape: {}'.format(cloth_info.shape))
print('cloth_info: \n{}'.format(cloth_info))

cloth_info shape: (11746, 326)
cloth_info: 
       frame         x0        y0        z0         vx0       vy0        vz0  \
0          0   0.140004 -48.67010  24.85760   -19.49826  2408.720 -1243.0920   
1          1   0.140004 -48.66920  24.85878   -19.49885  2408.374 -1243.4530   
2          2   0.140004 -48.66508  24.86367   -19.50129  2407.993 -1242.9610   
3          3   0.140004 -48.66156  24.86360   -19.50125  2408.227 -1243.0420   
4          4   0.140004 -48.67500  24.85834   -19.49862  2408.832 -1242.8460   
...      ...        ...       ...       ...         ...       ...        ...   
11741     37   5.389135 -46.39059  16.33694  -301.85690  2331.486  -920.2172   
11742     38 -40.357950 -39.44525  22.43649  1894.82300  1991.850 -1125.3890   
11743     39 -64.124770 -22.64233  25.18560  3139.49300  1207.923 -1285.2090   
11744     40 -32.848460 -42.10622  23.42051  1867.42700  1955.262 -1171.8740   
11745     41  33.562390 -42.82616  24.52554 -1533.56900  2171.362 -1210.6350

In [2]:
class ClothDataset(Dataset):
    def __init__(self, csv_data, num_vertices=25):
        """
        Args:
            csv_data (str or filepath): Path to the CSV file or raw CSV string.
            num_vertices (int): Number of vertices per frame.
        """
        # Load the CSV data into a pandas DataFrame
        #if isinstance(csv_data, str) and "frame,x0" in csv_data:
            #self.data = pd.read_csv(io.StringIO(csv_data.strip()))
        #else:
            #self.data = pd.read_csv(csv_data)
        self.data = csv_data
            
        #quick fix para espacios en primera fila
        self.data.columns = self.data.columns.str.strip()
        
        self.num_vertices = num_vertices
        
        # Define the base feature names to extract per vertex
        self.feature_prefixes = ['x', 'y', 'z', 'vx', 'vy', 'vz', 'sdf', 'nx', 'ny', 'nz', 'md', 'u', 'v']

        self.position_prefixes = ['x', 'y', 'z']
        self.output_positions = self.data.filter(regex=r'^[xyz]\d+$')

        # Quitamos primera fila de outputs (no es el output de nada) y ultima fila de input (no tiene output)
        self.output_positions = self.output_positions.iloc[1:]
        self.data = self.data.iloc[:-1]
        

    def __len__(self):
        # The number of items is the number of frames (rows) in the dataset
        return len(self.data)
    
    def num_features(self):
        # Number of columns in a row (pos, vel, sdf, uv per vertex)
        return len(self.feature_prefixes)
    
    def num_vertex(self):
        return self.num_vertices
    
    def _get_frame_output_tensor(self, idx):
         row = self.output_positions.iloc[idx]
         frame_data = []
         for i in range(self.num_vertices):
            vertex_cols = [f"{prefix}{i}" for prefix in self.position_prefixes]
            vertex_features = row[vertex_cols].values.astype(np.float32)
            frame_data.append(vertex_features)

         tensor_data = torch.tensor(np.array(frame_data))

         return tensor_data
    
    def _get_frame_tensor(self, idx):
        row = self.data.iloc[idx]
        frame_data = []

        for i in range(self.num_vertices):
            vertex_cols = [f"{prefix}{i}" for prefix in self.feature_prefixes]
            vertex_features = row[vertex_cols].values.astype(np.float32)
            frame_data.append(vertex_features)

        tensor_data = torch.tensor(np.array(frame_data))

        return tensor_data
        
    def __getitem__(self, idx):
        frame_t = self._get_frame_tensor(idx)
        frame_t1 = self._get_frame_output_tensor(idx) # +1 ya no TODO: Pillar solo las columnas de pos

        return frame_t, frame_t1

# --- Example Usage ---

# (Assuming 'csv_string' is a variable holding your provided data block)
dataset = ClothDataset(cloth_info)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

for batch_data, batch_frames in dataloader:
    print(f"Batch Shape: {batch_data.shape}")
    print(batch_data)
    print(batch_frames)
    break


mean = 0
std = 0
n_samples = 0

for batch_t, _ in dataloader:
    batch_t = batch_t.float()
    
    batch_samples = batch_t.size(0)
    batch_t = batch_t.view(-1, batch_t.size(-1))  

    mean += batch_t.mean(dim=0)
    std += batch_t.std(dim=0)
    n_samples += 1

mean /= n_samples
std /= n_samples

# evitamos division por 0
std[std < 1e-8] = 1.0

print("MEAN:", mean)
print("STD:", std)

Batch Shape: torch.Size([4, 25, 13])
tensor([[[-38.8403, -39.1042,  21.9697,  ...,   1.0000,   0.7500,   0.0000],
         [-33.1015, -15.2702,  47.0629,  ...,   1.0000,   1.0000,   0.2500],
         [-51.6412, -31.3168,  41.9793,  ...,   1.0000,   1.0000,   0.0000],
         ...,
         [-10.7469,  28.8504, -49.2165,  ...,   1.0000,   0.0000,   0.7500],
         [  0.1400,  51.3300, -25.1385,  ...,   0.0000,   0.2500,   1.0000],
         [  0.1400,  51.3300, -50.1385,  ...,   0.0000,   0.0000,   1.0000]],

        [[ 41.8677, -39.5740,  23.8370,  ...,   1.0000,   0.7500,   0.0000],
         [ 37.5320, -13.6095,  47.2526,  ...,   1.0000,   1.0000,   0.2500],
         [ 53.0105, -33.0530,  45.2475,  ...,   1.0000,   1.0000,   0.0000],
         ...,
         [  7.1092,  27.3280, -50.0167,  ...,   1.0000,   0.0000,   0.7500],
         [  0.1400,  51.3300, -25.1385,  ...,   0.0000,   0.2500,   1.0000],
         [  0.1400,  51.3300, -50.1385,  ...,   0.0000,   0.0000,   1.0000]],

       

In [6]:
import json
# Intento de normalización uep

# Convertir tensores a listas
norm_data = {
    "mean": mean.tolist(),
    "std": std.tolist(),
    "feature_prefixes": ['x', 'y', 'z', 'vx', 'vy', 'vz', 'sdf', 'nx', 'ny', 'nz', 'md', 'u', 'v']
}

with open("cloth_norm_params.json", "w") as f:
    json.dump(norm_data, f)

In [3]:
# TODO
# una recurrente sencilla (la salida se vuelve entrada en el siguiente ejemplo)
# Antes de meternos en CNN y LSTM
# AÑADIMOS VALORES U V PARA CADA VERTICE ( no queremos perder la noción espacial )

import torch.nn as nn
import torch.nn.functional as F #acceso rapido a funciones
import torch.utils.data as data #cargar y manejar el training data

class MyModule(nn.Module):

    def __init__(self, num_inputs, num_hidden, num_outputs):
        super().__init__()
        # Some init for my module
        self.linear1 = nn.Linear(num_inputs, num_hidden)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(num_hidden, num_outputs)

    def forward(self, x):
        # Function for performing the calculation of the module.
        
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        #print(x)
        return x

    #backward se hace automaticamente, podriamos definirla tmbn 

#tmbn clases DataSet y DataLoader

# definir modelo, loss function y optimizer
#TODO: buscar dimensiones reales de las neuronas

# nn.Linear in PyTorch is designed to handle 3D tensors seamlessly.  
# When a 3D input tensor (e.g., batch_size, sequence_length, features) is provided,
#  the layer applies the linear transformation only to the last dimension (the features dimension),
#  preserving all other dimensions.

model = MyModule(num_inputs=13, num_hidden= 64, num_outputs=3)
# print, save, lo que sea
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

for epoch in range(50):
    for batch_t, batch_t1 in dataloader:

        batch_t = batch_t.float()
        batch_t1 = batch_t1.float()

        batch_t = (batch_t - mean) / std

        # output (posiciones)
        mean_out = batch_t1.mean(dim=(0,1), keepdim=True)
        std_out = batch_t1.std(dim=(0,1), keepdim=True) + 1e-8
        batch_t1 = (batch_t1 - mean_out) / std_out

        pred = model(batch_t)

        loss = criterion(pred, batch_t1)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        print(f'Epoch {epoch+1}, Loss: {loss.item()}')


Epoch 1, Loss: 1.0131884813308716
Epoch 1, Loss: 1.0791919231414795
Epoch 1, Loss: 1.0251765251159668
Epoch 1, Loss: 1.0062682628631592
Epoch 1, Loss: 0.9779784083366394
Epoch 1, Loss: 1.0134782791137695
Epoch 1, Loss: 1.1233150959014893
Epoch 1, Loss: 1.0140223503112793
Epoch 1, Loss: 0.9994040727615356
Epoch 1, Loss: 0.9973675608634949
Epoch 1, Loss: 0.93170166015625
Epoch 1, Loss: 0.9193325042724609
Epoch 1, Loss: 0.9808840155601501
Epoch 1, Loss: 1.0062135457992554
Epoch 1, Loss: 0.9742093682289124
Epoch 1, Loss: 0.8911413550376892
Epoch 1, Loss: 0.9458044171333313
Epoch 1, Loss: 0.9174487590789795
Epoch 1, Loss: 0.9537660479545593
Epoch 1, Loss: 0.9044098854064941
Epoch 1, Loss: 0.9616506099700928
Epoch 1, Loss: 0.9058985114097595
Epoch 1, Loss: 0.9317241311073303
Epoch 1, Loss: 0.9071022272109985
Epoch 1, Loss: 0.9268044829368591
Epoch 1, Loss: 0.9411466717720032
Epoch 1, Loss: 0.9394246339797974
Epoch 1, Loss: 0.9557576775550842
Epoch 1, Loss: 0.9189133644104004
Epoch 1, Loss: 0

In [4]:
#INTENTO DE EXPORTAR A ONNX
import sys
print(sys.executable)

import onnx
import onnxruntime

print("ONNX version:", onnx.__version__)
print("ONNX Runtime version:", onnxruntime.__version__)

# Create example inputs for exporting the model. The inputs should be a tuple of tensors.
example_inputs = (batch_t)
onnx_program = torch.onnx.export(model, example_inputs, dynamo=True)

onnx_program.save("onnxModels/trainedModel.onnx")

c:\Users\Eva\miniconda3\envs\dl2024\python.exe
ONNX version: 1.21.0
ONNX Runtime version: 1.24.4
[torch.onnx] Obtain model graph for `MyModule([...]` with `torch.export.export`...
[torch.onnx] Obtain model graph for `MyModule([...]` with `torch.export.export`... ✅
[torch.onnx] Translate the graph into ONNX...


c:\Users\Eva\miniconda3\envs\dl2024\Lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5060 Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90 sm_37 compute_37.
If you want to use the NVIDIA GeForce RTX 5060 Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(
W0417 02:26:31.321000 37704 site-packages\torch\onnx\_internal\exporter\_schemas.py:446] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0417 02:26:31.322000 37704 site-packages\torch\onnx\_internal\exporter\_schemas.py:446] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0,

[torch.onnx] Translate the graph into ONNX... ✅


In [5]:
import torch
import torch.nn as nn

#EJEMPLO SENCILLO CONVOLUCIONAL PARA MÁS ADELANTE

# Example: 100 features, 1 channel (linear input)
# Batch size=16
input_data = torch.randn(16, 1, 100) 

model = nn.Sequential(
    nn.Conv1d(in_channels=1, out_channels=32, kernel_size=3), # Extract features
    nn.ReLU(),
    nn.Flatten(), # Flatten for Dense layer
    nn.Linear(32 * 98, 10) # 98 is the new length after convolution
)
